# Tier 03 — Learn

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/f-inverse/jammi-ai/blob/py-v0.49.1/cookbook/notebooks/book/03-learn/learn.ipynb)

Built from [`cookbook/book/chapters/03-learn/learn.qmd`](https://github.com/f-inverse/jammi-ai/blob/main/cookbook/book/chapters/03-learn/learn.qmd). Run the setup
cell first; every other cell runs top to bottom.

In [ ]:
# Setup: jammi 0.49.1 — the CUDA engine on an sm_80+ GPU (L4, A100, …), the
# CPU engine otherwise — and the cookbook's library and fixtures. The chapter runs
# at `small` scale, over the committed fixtures, in minutes. SCALE = "full" runs
# it over the published data and real encoders instead: meant for a GPU, and the
# chapters that fine-tune take hours there.
import os
import subprocess
import sys


def compute_capability() -> float:
    try:
        out = subprocess.run(
            ["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"],
            capture_output=True, text=True, check=True,
        ).stdout.split()
    except (OSError, subprocess.CalledProcessError):
        return 0.0
    return float(out[0]) if out else 0.0


gpu = compute_capability() >= 8.0
engine = "jammi-ai-native-cu12" if gpu else "jammi-ai-native"
server = "jammi-server-cu12" if gpu else "jammi-server"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jammi-ai==0.49.1", engine + "==0.49.1", "jammi-cookbook==0.49.1"], check=True)
SCALE = "small"
os.environ["JAMMI_COOKBOOK_SCALE"] = SCALE
print(f"engine: {engine}   scale: {SCALE}")

In [ ]:
import jammi_cookbook

**Recipe:** `fine_tune_graph` · `eval_compare` · **Theory:** representation
learning on graphs — node2vec / contrastive (Stanković et al. 2020, Part III;
Hamilton 2017); citation-graph contrastive supervision (SPECTER, Cohan et al.
2020); nearest-neighbor-of-own-representations (NNCLR, Dwibedi et al. 2021) ·
**Rail:** measurement (declared-edge gain, and two negative controls).

Tier 02 *propagated* fixed embeddings; tier 03 **learns** new ones.
`fine_tune_graph` samples random walks over a graph and fine-tunes the encoder
contrastively: papers that co-occur on a walk are pulled together, others
pushed apart. The graph supervises the metric — but *which* graph matters, and
this chapter measures that directly by training the identical recipe over
three graphs:

- **declared** — the citation graph: who cited whom, a signal external to the
  encoder;
- **similarity** — tier 01's k-NN graph over the encoder's own embeddings
  (`edge_provenance="similarity"`);
- **random** — uniform random pairs over the same papers: structure-free.

In [ ]:
import tempfile

import jammi
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
from jammi_cookbook import contracts, datasets, keystone, scale

SCALE = scale.current()
db = jammi.connect(f"file://{tempfile.mkdtemp()}")
arxiv = datasets.arxiv(db, SCALE)
base = keystone.embed(db, arxiv, SCALE)

The step, run once per graph:

In [ ]:
keystone.show(keystone.fine_tune_on_graph)

## The declared graph

In [ ]:
declared = keystone.fine_tune_on_graph(
    db, arxiv, SCALE, edge_source=arxiv.cites, provenance="declared",
    epochs=keystone.FINE_TUNE_EPOCHS[SCALE],
)
print("fine-tuned embeddings:", declared)

## The two controls

The similarity graph is the engine's own `build_neighbor_graph` table over the
base embeddings, which the fine-tune walks directly as `edge_graph_table` — the
training set then pins that table by its content digest, as any derived table
pins a result-table input. The random graph has as many edges as the similarity
graph, drawn uniformly over the same papers, and is registered as a source.

In [ ]:
similarity_graph = db.build_neighbor_graph(arxiv.papers, embedding_table=base, k=10, exact=True)

ids = np.array(arxiv.split["train"] + arxiv.split["valid"] + arxiv.split["test"])
n_edges = db.sql(f'SELECT COUNT(*) AS n FROM "jammi.{similarity_graph}"').to_pylist()[0]["n"]
rng = np.random.default_rng(0)
random_path = f"{tempfile.mkdtemp()}/random_edges.parquet"
pq.write_table(
    pa.table({"src": rng.choice(ids, n_edges).tolist(), "dst": rng.choice(ids, n_edges).tolist()}),
    random_path,
)
db.add_source("random_edges", url=random_path, format="parquet")

similarity = keystone.fine_tune_on_graph(
    db, arxiv, SCALE, edge_graph_table=similarity_graph, provenance="similarity",
    epochs=keystone.CONTROL_EPOCHS[SCALE],
)
random_graph = keystone.fine_tune_on_graph(
    db, arxiv, SCALE, edge_source="random_edges", provenance="declared",
    epochs=keystone.CONTROL_EPOCHS[SCALE],
)

## The measurement rail: which graph supervises the metric?

One `eval_compare` over all four tables — the base encoder first, so every
other entry carries its delta against it — on tier 01's same-subject golden.

In [ ]:
golden = keystone.subject_golden(db, arxiv)
compared = db.eval_compare(
    embedding_tables=[base, declared, similarity, random_graph],
    source=arxiv.papers, golden_source=golden, k=10,
)
arms = ["base", "declared", "similarity", "random"]
precision, gain = {}, {}
for arm, entry in zip(arms, compared["per_table"]):
    precision[arm] = entry["embedding_eval"]["aggregate"]["precision_at_k"]
    gain[arm] = entry["delta"]["precision_at_k"]["absolute"] if entry["delta"] else 0.0
    print(f"{arm:<11} precision@10 {precision[arm]:.3f}   Δ vs base {gain[arm]:+.3f}")

In [ ]:
for arm in arms:
    contracts.assert_close(f"arxiv.tier03.{arm}_precision_at_10", precision[arm], tol=0.02)
if SCALE is scale.Scale.FULL:
    # The finding, at the scale it is about: random hurts, a self-similarity
    # bootstrap helps a little, the declared graph helps most.
    assert gain["random"] < 0 < gain["similarity"] < gain["declared"]

At `full` scale the ordering is **random (harmful) < base < similarity (real,
partial) < declared (largest)**:

- **declared** — the largest gain: an external signal the encoder did not
  already have, trained toward convergence;
- **similarity** — a smaller but real gain. Training against a paper's own
  nearest neighbours pulls same-subject papers closer through the same
  homophily mechanism nearest-neighbour contrastive learning exploits (NNCLR,
  Dwibedi et al. 2021; SciNCL's citation-embedding k-NN augmentation,
  Ostendorff et al. 2022) — a weak bootstrap, not nothing;
- **random** — harmful: training against structureless pairs degrades the
  metric rather than merely failing to improve it.

At `small` scale all three runs are two epochs on a random-weight encoder, so
their numbers check the pipeline, not the finding; run this chapter at `full`
scale to see the ordering, measured.

In [ ]:
db.close()

## Bridge note (a signature chapter — deepened in the bridge chapters)

> **Graph-supervised metric learning = contrastive fine-tune over walk-sampled
> graph edges.** node2vec / DeepWalk's skip-gram over random walks (Grover &
> Leskovec 2016; the monograph's Part III graph embeddings; Hamilton et al. 2017)
> ↔ `fine_tune_graph`. The walk sampler turns a graph into positive pairs; the
> contrastive loss is the learning objective; `edge_provenance` names *which*
> graph — declared external structure (SPECTER, Cohan et al. 2020) supervises
> the largest gain, self-similarity (NNCLR, Dwibedi et al. 2021; SciNCL,
> Ostendorff et al. 2022) a smaller real one, and a random graph none at all.